<a href="https://colab.research.google.com/github/Nafiaakter/CSE707/blob/main/task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, NGram
from pyspark.ml import Pipeline
from pyspark.ml.linalg import DenseVector

# Step 1: Initialize Spark
spark = SparkSession.builder.appName("TFIDF-Example").getOrCreate()

# Step 2: Sample dataset
data = [
    (0, "Apache Spark is a powerful distributed computing engine for big data processing."),
    (1, "Big data text mining needs scalable and efficient frameworks."),
    (2, "TF-IDF is commonly used for feature extraction in text classification."),
    (3, "Machine learning models need high-quality features from text data."),
]

df = spark.createDataFrame(data, ["id", "text"])
df.show(truncate=False)

# Step 3: Define preprocessing steps
tokenizer = Tokenizer(inputCol="text", outputCol="words_token")
remover = StopWordsRemover(inputCol="words_token", outputCol="words_clean")


ngram = NGram(n=3, inputCol="words_clean", outputCol="trigrams")


vectorizer = CountVectorizer(inputCol="words_clean", outputCol="rawFeatures")

idf = IDF(inputCol="rawFeatures", outputCol="features")

# Step 4: Create and run the pipeline
pipeline = Pipeline(stages=[tokenizer, remover, ngram, vectorizer, idf])
model = pipeline.fit(df)
result = model.transform(df)

# Step 5: Show the results
result.select("id", "words_clean", "trigrams", "features").show(truncate=False)



def top_terms(features, vocab):
    pairs = list(enumerate(features.toArray()))
    sorted_terms = sorted(pairs, key=lambda x: -x[1])[:5]
    return [(vocab[idx], round(score, 3)) for idx, score in sorted_terms if score > 0]

vocab = model.stages[3].vocabulary

result.select("id", "features").rdd.map(
    lambda row: (row['id'], top_terms(row['features'], vocab))
).collect()

for doc_id, keywords in top_keywords_per_doc:
    print(f"Document {doc_id} Top Terms: {keywords}")



+---+--------------------------------------------------------------------------------+
|id |text                                                                            |
+---+--------------------------------------------------------------------------------+
|0  |Apache Spark is a powerful distributed computing engine for big data processing.|
|1  |Big data text mining needs scalable and efficient frameworks.                   |
|2  |TF-IDF is commonly used for feature extraction in text classification.          |
|3  |Machine learning models need high-quality features from text data.              |
+---+--------------------------------------------------------------------------------+

+---+---------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------